In [ ]:
import sys
import os

sys.path.append(os.path.abspath(".."))

In [ ]:
import os
from tqdm import tqdm
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from losses.final_loss import InverseDistanceBoundaryDiceLoss
from models.cbam_nnunet import nnUNet3D_CBAM
from utils.unets_helper_functions import (
    set_seed,
    save_checkpoint,
    final_PatchDataset_cbam,
    evaluate_full_volume_cbam,
    compute_per_class_dice
    
)

In [ ]:
set_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


In [ ]:
FOLD = 0

with open(f"../data/splits/fold_{FOLD}/train.txt") as f:
    train_cases = f.read().splitlines()

with open(f"../data/splits/fold_{FOLD}/val.txt") as f:
    val_cases = f.read().splitlines()

print("Train cases:", len(train_cases))
print("Val cases:", len(val_cases))


In [ ]:
PATCH_SIZE = 96
PATCHES_PER_CASE = 12
FOREGROUND_PROB = 0.7
NUM_WORKERS = 2
train_dataset = final_PatchDataset_cbam(train_cases,
                        "../data/processed/imagesTr",
                        "../data/processed/labelsTr",
                        patches_per_case = PATCHES_PER_CASE,
                        patch_size = PATCH_SIZE,
                        foreground_prob = FOREGROUND_PROB,
                        augment=True)

val_dataset = final_PatchDataset_cbam(val_cases,
                        "../data/processed/imagesTr",
                        "../data/processed/labelsTr",
                        patches_per_case = 1,
                        patch_size = PATCH_SIZE,
                        foreground_prob = FOREGROUND_PROB,
                        augment=False)


train_loader = DataLoader(
    train_dataset,
    batch_size=2,
    shuffle=True,
    num_workers=NUM_WORKERS,      
    pin_memory=True    
)

val_loader = DataLoader(
    val_dataset,
    batch_size=1,
    num_workers=NUM_WORKERS,    
    pin_memory=True
)

In [ ]:
model = nnUNet3D_CBAM(
    in_channels=1,
    out_channels=7,
    base_filters=24   # or try 32
).to(device)

print("cbam nnU-Net style model Initialized")


In [ ]:
EPOCHS = 100

optimizer = torch.optim.AdamW(model.parameters(), lr=2e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=EPOCHS
)

weights = torch.tensor([0.05, 1, 1, 1.2, 1.8, 1.8, 1]).to(device)
loss_fn = InverseDistanceBoundaryDiceLoss(
    epsilon=1e-5,
    lambda_weight=0.6,
    ce_weights = weights
)
scaler = torch.cuda.amp.GradScaler()


In [ ]:
def train_one_epoch_cbam(model, loader, optimizer, scaler, loss_fn, device):

    model.train()
    total_loss = 0

    for images, labels, dist_maps in tqdm(loader):

        images = images.to(device)
        labels = labels.long().to(device)
        dist_maps = dist_maps.to(device)

        optimizer.zero_grad()

        with torch.amp.autocast("cuda"):

            outputs = model(images)

            # Deep supervision unpack
            out, ds2, ds3, ds4 = outputs


            loss_main = loss_fn(out, labels, dist_maps)
            loss_ds2  = loss_fn(ds2, labels, dist_maps)
            loss_ds3  = loss_fn(ds3, labels, dist_maps)
            loss_ds4  = loss_fn(ds4, labels, dist_maps)

            loss = (
                1.0 * loss_main +
                0.5 * loss_ds2 +
                0.25 * loss_ds3 +
                0.125 * loss_ds4
            )

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()

    return total_loss / len(loader)

def validate_one_epoch_cbam(model, loader, loss_fn, device, num_classes=7):

    model.eval()
    total_loss = 0
    all_dices = []

    with torch.no_grad():
        for images, labels, dist_maps in loader:

            images = images.to(device)
            labels = labels.long().to(device)
            dist_maps = dist_maps.to(device)

            with torch.amp.autocast("cuda"):

                outputs = model(images)
                out = outputs[0]

                loss = loss_fn(out, labels, dist_maps)

            total_loss += loss.item()

            batch_dice = compute_per_class_dice(out, labels, num_classes)
            all_dices.append(batch_dice)

    mean_loss = total_loss / len(loader)

    all_dices = np.array(all_dices)
    mean_dices = np.mean(all_dices, axis=0)

    return mean_loss, mean_dices

In [ ]:
PROJECT_ROOT = os.path.abspath("..")
SAVE_DIR = os.path.join(PROJECT_ROOT, "experiments", "final_cbam_fold0")

os.makedirs(SAVE_DIR, exist_ok=True)

best_val_loss = float("inf")
best_dice = 0.0
best_parotid = 0.0

for epoch in range(EPOCHS):

    # ------------------ TRAIN ------------------
    train_loss = train_one_epoch_cbam(
        model,
        train_loader,
        optimizer,
        scaler,
        loss_fn,
        device
    )

    # ------------------ VALIDATE ------------------
    val_loss, val_dice = validate_one_epoch_cbam(
        model,
        val_loader,
        loss_fn,
        device
    )

    mean_dice = val_dice.mean()
    parotid_score = (val_dice[3] + val_dice[4]) / 2

    print(f"\nEpoch {epoch}")
    print(f"Train Loss: {train_loss:.4f}")
    print(f"Val Loss: {val_loss:.4f}")
    print(f"Per Class Dice: {val_dice}")
    print(f"Mean Dice: {mean_dice:.4f}")
    print(f"Parotid Dice (L,R): {val_dice[3]:.4f}, {val_dice[4]:.4f}")

    # ------------------ SAVE BEST LOSS ------------------
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        save_checkpoint(
            model,
            optimizer,
            epoch,
            best_val_loss,
            os.path.join(SAVE_DIR, "best_loss_model.pth")
        )
        print("Saved Best Loss Model")

    # ------------------ SAVE BEST DICE ------------------
    if mean_dice > best_dice:
        best_dice = mean_dice
        save_checkpoint(
            model,
            optimizer,
            epoch,
            best_val_loss,
            os.path.join(SAVE_DIR, "best_dice_model.pth")
        )
        print("Saved Best Dice Model")

    # ------------------ SAVE BEST PAROTID ------------------
    if parotid_score > best_parotid:
        best_parotid = parotid_score
        save_checkpoint(
            model,
            optimizer,
            epoch,
            best_val_loss,
            os.path.join(SAVE_DIR, "best_parotid_model.pth")
        )
        print("Saved Best Parotid Model")

    scheduler.step()